In [0]:
-----------------------------------------------------------------------------------------
---CHECKING THE USER PROFILE DATASET
-----------------------------------------------------------------------------------------

SELECT * 
FROM `workspace`.`bright_tv`.`user_profiles_tv` 
LIMIT 1000;

----------------------------------------------------------------------------------------
---CHECKING THE VIEWERSHIP DATASET
----------------------------------------------------------------------------------------

SELECT * 
FROM `workspace`.`bright_tv`.`viewership_tv` 
LIMIT 1000;

----------------------------------------------------------------------------------------
---SELECT DISTINCT
----------------------------------------------------------------------------------------

SELECT DISTINCT Channel2
FROM `workspace`.`bright_tv`.`viewership_tv` 
LIMIT 1000;

----------------------------------------------------------------------------------------
---FULL JOIN
----------------------------------------------------------------------------------------

SELECT *
FROM `workspace`.`bright_tv`.`user_profiles_tv` 
FULL JOIN `workspace`.`bright_tv`.`viewership_tv` 
      ON `workspace`.`bright_tv`.`user_profiles_tv`.UserID = `workspace`.`bright_tv`.`viewership_tv`.UserID0;

----------------------------------------------------------------------------------------
---LONG QUERY 
----------------------------------------------------------------------------------------

SELECT 
       COALESCE(up.UserID, vw.UserID0) AS UserID,
       COALESCE(up.Name, 'Unknown') AS Name,
       COALESCE(up.Surname, 'Unknown') AS Surname,
       COALESCE(up.Gender, 'Unknown') AS Gender,
       COALESCE(up.Race, 'Unknown') AS Race,
       COALESCE(up.Age, 0) AS Age,
       COALESCE(up.Province, 'Unknown') AS Province,
       COALESCE(vw.Channel2, 'Unknown') AS Channel2,

-------------------------------------------------------------------------------------------------------------------------
-- CONVETING TIME INTO RSA TIME (SAST)
-------------------------------------------------------------------------------------------------------------------------

       from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg') AS RecordDate_RSA,

       date_format(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg'), 'EEEE') AS Day_Name,

       DAY(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg')) AS Day,
       MONTH(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg')) AS Month,
       YEAR(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg')) AS Year,

       date_format(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg'),'HH:mm:ss') AS Time,
       DATE_FORMAT(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg'), 'yyyy-MM-dd') AS Date,
--------------------------------------------------------------------------------------------------------------------------
-- CREATING A COLUMN FOR TIME FRAME
--------------------------------------------------------------------------------------------------------------------------

       CASE
           WHEN date_format(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg'),'HH:mm:ss') BETWEEN '00:00:00' AND '11:59:59' THEN 'Morning'
           WHEN date_format(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg'),'HH:mm:ss') BETWEEN '12:00:00' AND '17:59:59' THEN 'Afternoon'
           WHEN date_format(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg'),'HH:mm:ss') BETWEEN '18:00:00' AND '21:59:59' THEN 'Evening'
           WHEN date_format(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg'),'HH:mm:ss') >= '22:00:00' THEN 'Late'
       END AS Time_Frame,

--------------------------------------------------------------------------------------------------------------------------
-- CREATING COLUMN FOR AGE BUCKET
--------------------------------------------------------------------------------------------------------------------------

       CASE
           WHEN COALESCE(up.Age,0) BETWEEN 0 AND 14 THEN 'Child'
           WHEN COALESCE(up.Age,0) BETWEEN 15 AND 25 THEN 'Youth'
           WHEN COALESCE(up.Age,0) BETWEEN 26 AND 50 THEN 'Adult'
           ELSE 'Elderly'
       END AS Age_Bucket,

--------------------------------------------------------------------------------------------------------------------------
-- CREATING A COLUMN FOR CHANNEL CATEGORY
--------------------------------------------------------------------------------------------------------------------------

       CASE
           WHEN vw.Channel2 IN ('ICC Cricket World Cup 2011', 'Supersport Live Events', 'SuperSport Blitz', 'Live on SuperSport') THEN 'Sports'
           WHEN vw.Channel2 IN ('Trace TV', 'Channel O') THEN 'Music'
           WHEN vw.Channel2 IN ('SawSee', 'E! Entertainment', 'Break in transmission', 'kykNET', 'Africa Magic', 'Vuzu', 'M-Net', 'DStv Events 1', 'MK', 'Sawsee', 'Wimbledon', 'CNN') THEN 'Entertainment'
           WHEN vw.Channel2 IN ('Boomerang', 'Cartoon Network') THEN 'Cartoons'
           ELSE 'Other'
       END AS Channel_Category,
    
----------------------------------------------------------------------------------------------------------------------------
-- COUNTING THE TOTAL NUMBER OF VIEWERS 
----------------------------------------------------------------------------------------------------------------------------

       COUNT(*) AS View_Count

----------------------------------------------------------------------------------------------------------------------------
-- FROM & FULL JOIN
----------------------------------------------------------------------------------------------------------------------------

FROM `workspace`.`bright_tv`.`user_profiles_tv` up
FULL JOIN `workspace`.`bright_tv`.`viewership_tv` vw
      ON up.UserID = vw.UserID0

----------------------------------------------------------------------------------------------------------------------------
-- GROUP BY STATEMENT
----------------------------------------------------------------------------------------------------------------------------

GROUP BY 
       COALESCE(up.UserID, vw.UserID0),
       COALESCE(up.Name, 'Unknown'),
       COALESCE(up.Surname, 'Unknown'),
       COALESCE(up.Gender, 'Unknown'),
       COALESCE(up.Race, 'Unknown'),
       COALESCE(up.Age, 0),
       COALESCE(up.Province, 'Unknown'),
       COALESCE(vw.Channel2, 'Unknown'),
       from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg'),
       DAY(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg')),
       MONTH(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg')),
       YEAR(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg')),
       date_format(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg'),'HH:mm:ss'),
       DATE_FORMAT(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg'), 'yyyy-MM-dd'),
       CASE
           WHEN date_format(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg'),'HH:mm:ss') BETWEEN '00:00:00' AND '11:59:59' THEN 'Morning'
           WHEN date_format(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg'),'HH:mm:ss') BETWEEN '12:00:00' AND '17:59:59' THEN 'Afternoon'
           WHEN date_format(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg'),'HH:mm:ss') BETWEEN '18:00:00' AND '21:59:59' THEN 'Evening'
           WHEN date_format(from_utc_timestamp(vw.RecordDate2, 'Africa/Johannesburg'),'HH:mm:ss') >= '22:00:00' THEN 'Late'
       END,
       CASE
           WHEN COALESCE(up.Age,0) BETWEEN 0 AND 14 THEN 'Child'
           WHEN COALESCE(up.Age,0) BETWEEN 15 AND 25 THEN 'Youth'
           WHEN COALESCE(up.Age,0) BETWEEN 26 AND 50 THEN 'Adult'
           ELSE 'Elderly'
       END,
       CASE
           WHEN vw.Channel2 IN ('ICC Cricket World Cup 2011', 'Supersport Live Events', 'SuperSport Blitz', 'Live on SuperSport') THEN 'Sports'
           WHEN vw.Channel2 IN ('Trace TV', 'Channel O') THEN 'Music'
           WHEN vw.Channel2 IN ('SawSee', 'E! Entertainment', 'Break in transmission', 'kykNET', 'Africa Magic', 'Vuzu', 'M-Net', 'DStv Events 1', 'MK', 'Sawsee', 'Wimbledon', 'CNN') THEN 'Entertainment'
           WHEN vw.Channel2 IN ('Boomerang', 'Cartoon Network') THEN 'Cartoons'
           ELSE 'Other'
       END

---------------------------------------------------------------------------------------------------------------------------
-- ORDER BY STATEMENT 
---------------------------------------------------------------------------------------------------------------------------

ORDER BY 
       Year ASC,
       Month ASC,
       Day ASC,
       UserID ASC;


---------------------------------------------------------------------------------------------------------------------------